In [1]:
import os
import requests
import json
from typing import Optional, Dict, Any
import pandas as pd
from dotenv import load_dotenv

from utils import DuckdbUtils
from dags.data_collector import CryptoDataCollector

1. raw layer

In [2]:
sample_asset = 'BTC'

sample_data_collector = CryptoDataCollector(
    sample_asset,
    'iceberg.cryptocurrencies_project_raw.btc',
    'iceberg.cryptocurrencies_project_raw_stg.btc',
)

In [3]:
sample_result_row_cnt = sample_data_collector.process()

/opt/workspace/ETL/dags/data_collector.py:54: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  pdf = pd.read_json(json_data_str, orient='index')


In [4]:
print(f"Successfully loaded {sample_result_row_cnt} rows")

Successfully loaded 5714 rows


2. dds layer

In [1]:
import os
from pyspark.sql import SparkSession
os.environ['SPARK_CONF_DIR'] = '/opt/spark/conf'

from utils import PySparkTableLoader

In [2]:
spark = SparkSession \
    .builder \
    .appName("union_cryptoasset_data") \
    .master(os.getenv('SPARK_MASTER_URL')) \
    .config('spark.ui.port', '4040') \
    .config('spark.driver.cores', '1') \
    .config('spark.driver.memory', '1G') \
    .config('spark.executor.instances', '1') \
    .config('spark.executor.cores', '1') \
    .config('spark.executor.memory', '1G') \
    .config('spark.sql.catalog.iceberg.s3.connection-maximum', '50') \
    .config('spark.sql.catalog.iceberg.s3.threads-max', '10') \
    .config('spark.sql.catalog.iceberg.s3.fast-upload ', 'true') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
spark.conf.set('spark.sql.shuffle.partitions', '10')

26/03/14 18:08:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [18]:
DDS_MAX_DATE = spark.sql("SELECT COALESCE(MAX(dt) - interval '7 days', '2020-01-01') FROM iceberg.cryptocurrencies_project_dds.trade_data").collect()[0][0]

In [4]:
DDS_QUERY = f"""
    SELECT asset
        , dt
        , open_price
        , high_price
        , low_price
        , close_price
        , trading_volume
        , src_processed_dttm
        , processed_dttm
    FROM (
        SELECT *
            , ROW_NUMBER() OVER(PARTITION BY asset, dt ORDER BY src_processed_dttm DESC) AS rn
        FROM (
            SELECT 'BTC' AS asset
                    , dt
                    , open_price
                    , high_price
                    , low_price
                    , close_price
                    , trading_volume
                    , processed_dttm AS src_processed_dttm
                    , CURRENT_TIMESTAMP AS processed_dttm
            FROM iceberg.cryptocurrencies_project_raw.btc
            WHERE dt >= '{DDS_MAX_DATE}'
            UNION ALL
            SELECT 'ETH' AS asset
                    , dt
                    , open_price
                    , high_price
                    , low_price
                    , close_price
                    , trading_volume
                    , processed_dttm AS src_processed_dttm
                    , CURRENT_TIMESTAMP AS processed_dttm
            FROM iceberg.cryptocurrencies_project_raw.eth
            WHERE dt >= '{DDS_MAX_DATE}'
        ) AS united_data
    ) AS dedubled_data
    WHERE rn = 1
    ;
"""

In [5]:
dds_table_loader = PySparkTableLoader(
    table_name='iceberg.cryptocurrencies_project_dds.trade_data',
    stg_table_name='iceberg.cryptocurrencies_project_dds_stg.trade_data',
    partition_by=['asset', 'dt'],
    spark_session=spark,
)

In [6]:
dds_table_loader\
    .calc_stg(DDS_QUERY)\
    .load_trg_scd1(overwrite_trg=False)

3. dm layer

In [8]:
spark.stop()

In [12]:
pd.set_option("max_colwidth", 1000)